# Formula-SWCT Generalization on FGVC-Aircraft

This notebook evaluates the final Formula-SWCT method on the FGVC-Aircraft fine-grained image-classification dataset using an ImageNet-pretrained ResNet-18.

FGVC-Aircraft introduces a challenging fine-grained recognition setting that differs substantially from the three Formula-SWCT calibration datasets:

- Beans
- DTD
- Flowers102

The evaluation follows the final predict-first, train-once protocol:

1. Collect stage statistics from the target training split using the frozen pretrained network.
2. Predict four stage-specific beta values using the frozen Formula-SWCT predictor.
3. Freeze the predicted beta values.
4. Extract fixed train, validation, and test features once.
5. Train one downstream linear classifier.
6. Select the classifier checkpoint using validation accuracy.
7. Evaluate once on the test split.

No FGVC-Aircraft-specific beta sweep or beta optimization is performed.

## Environment Setup

This notebook is designed to run independently in Google Colab.

The repository is cloned or updated, the `stage-wise-ct` branch is selected, and the research dependencies are installed.

After dependency installation, restart the Colab runtime before continuing.

In [ ]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

### Runtime Restart Required

Restart the Colab runtime after dependency installation.

After restarting, do not rerun the installation cell. Continue from the next cell.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

## FGVC-Aircraft Dataset Setup

The FGVC-Aircraft train, validation, and test loaders are constructed using the project's standard transfer-learning pipeline.

This cell verifies that the dataset can be loaded correctly before running the evaluation methods.

In [ ]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_fgvc-aircraft",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

# Baseline and Original S-CT Reference

The original Baseline and Single-Parameter Curvature Tuning methods are included as reference methods.

The Baseline uses the original frozen ImageNet-pretrained ResNet-18 representation.

S-CT searches for one global beta value shared across all curvature-modified activations using the original validation-based beta-selection procedure.

The original Aircraft experiment did not complete this comparison, so this section is retained for later execution to obtain a fair reference against Formula-SWCT.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

# Fit the Frozen Formula-SWCT Predictor

Because this notebook is designed to run independently in a fresh Google Colab runtime, the Formula-SWCT predictor is reconstructed before evaluating FGVC-Aircraft.

The formula is fitted using only the three fixed development/calibration datasets:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

For each calibration dataset and seed, the frozen ImageNet-pretrained ResNet-18 is used to collect five stage descriptors:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

These observations are used to fit the fixed ridge-regression Formula-SWCT predictor with:

**lambda = 0.10**

This fitting step is performed before loading FGVC-Aircraft for Formula-SWCT evaluation. FGVC-Aircraft does not modify the fitted formula.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode fit \
    --model resnet18 \
    --pretrained_ds imagenet \
    --train_bs 32 \
    --test_bs 64

# Final Formula-SWCT Evaluation

The frozen Formula-SWCT predictor is now applied to FGVC-Aircraft.

The predictor first measures the stage statistics of the frozen ImageNet-pretrained ResNet-18 using only the Aircraft training split.

The four stage-specific beta values are then predicted before downstream classifier training.

After prediction:

- all four beta values are frozen,
- train/validation/test features are extracted once,
- one downstream linear classifier is trained,
- validation accuracy selects the classifier checkpoint,
- the selected classifier is evaluated once on the test split.

No Aircraft-specific beta sweep or beta optimization is performed.

In [ ]:
!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

## Previously Completed Formula-SWCT Run

A Formula-SWCT FGVC-Aircraft seed-42 experiment was completed before the original notebook data was lost.

The recorded result was:

- Predicted beta vector: **[0.899803, 0.887323, 0.889689, 0.863769]**
- Best validation accuracy: **37.47%**
- Best validation epoch: **19**
- Test accuracy: **38.10%**

This result was obtained using the frozen Formula-SWCT predictor before an Aircraft S-CT comparison was completed.

The experiment will be reproduced using the execution cell above so that the result and generated JSON file are restored in this notebook.

# FGVC-Aircraft Results Summary

The Formula-SWCT seed-42 result currently available is:

| Method | Beta Selection | Test Accuracy |
|---|---|---:|
| Formula-SWCT | Direct four-stage prediction | **38.10%** |

Formula-SWCT predicted:

**[0.899803, 0.887323, 0.889689, 0.863769]**

with a best validation accuracy of **37.47%** at epoch **19**.

The Baseline and original S-CT rows will be added after the reference experiment in this notebook is executed.

No conclusions about superiority over S-CT should be drawn until that comparison has been completed.